In [3]:
# ==============================================================================
# ΜΗΝΑΣ 4: ΔΗΜΙΟΥΡΓΙΑ ΣΤΑΤΙΚΩΝ ΓΡΑΦΩΝ (GRAPH A & GRAPH B)
# ==============================================================================

import pandas as pd
import numpy as np
import torch
import networkx as nx
import matplotlib.pyplot as plt
import warnings
from statsmodels.tsa.api import VAR
from torch_geometric.utils import dense_to_sparse

warnings.filterwarnings('ignore')

# 1. Φόρτωση Κύριου Dataset (Yahoo Finance) - 10 νομίσματα
df_yahoo = pd.read_csv('crypto_log_returns_yahoo.csv', index_col='Date', parse_dates=True)
# Γεμίζουμε τα πιθανά NaNs με 0
df_yahoo = df_yahoo.fillna(0)

print(f"Δεδομένα Yahoo φορτώθηκαν: {df_yahoo.shape}")
num_nodes = df_yahoo.shape[1]

# --- Graph A: Correlation Graph (Pearson) ---
print("\n--- Υπολογισμός Graph A: Correlation Graph ---")
corr_matrix = df_yahoo.corr(method='pearson').values
# Thresholding: Κρατάμε ακμές μόνο με συσχέτιση > 0.4
adj_corr = np.where(np.abs(corr_matrix) > 0.4, corr_matrix, 0)
np.fill_diagonal(adj_corr, 0)
edge_index_corr, edge_weight_corr = dense_to_sparse(torch.tensor(adj_corr, dtype=torch.float32))

# --- Graph B: Volatility Spillover Graph (VAR) ---
print("--- Υπολογισμός Graph B: Spillover Graph (VAR) ---")
# Εκπαίδευση ενός γρήγορου VAR(1) για να βρούμε τα spillovers (πώς το coin A επηρεάζει το coin B)
var_model = VAR(df_yahoo)
var_results = var_model.fit(1)
spillover_matrix = var_results.coefs[0]
# Κρατάμε μόνο τις σημαντικές επιδράσεις (> 0.05)
adj_spill = np.where(np.abs(spillover_matrix) > 0.05, spillover_matrix, 0)
np.fill_diagonal(adj_spill, 0)
edge_index_spill, edge_weight_spill = dense_to_sparse(torch.tensor(adj_spill, dtype=torch.float32))

print(f"Correlation Edges: {edge_index_corr.shape[1]} | Spillover Edges: {edge_index_spill.shape[1]}")

Δεδομένα Yahoo φορτώθηκαν: (2986, 10)

--- Υπολογισμός Graph A: Correlation Graph ---
--- Υπολογισμός Graph B: Spillover Graph (VAR) ---
Correlation Edges: 84 | Spillover Edges: 24


In [4]:
# ==============================================================================
# ΠΡΟΕΤΟΙΜΑΣΙΑ 3D TENSORS ΓΙΑ ΤΑ ST-GNNs
# ==============================================================================
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import DataLoader, TensorDataset

# Στόχος: Η Απόλυτη Απόδοση (Μεταβλητότητα) όλων των 10 νομισμάτων
volatility_data = np.abs(df_yahoo.values)

# Scaling 0-1
scaler = MinMaxScaler()
volatility_scaled = scaler.fit_transform(volatility_data)

# Δημιουργία X, y (Κοιτάμε 14 μέρες πίσω για να προβλέψουμε την επόμενη για ΟΛΑ τα coins)
SEQ_LENGTH = 14

def create_graph_sequences(data, seq_len):
    xs, ys = [], []
    for i in range(len(data) - seq_len):
        x = data[i : i + seq_len]     # Shape: [14, 10]
        y = data[i + seq_len]         # Shape: [10]
        xs.append(x)
        ys.append(y)
    return np.array(xs), np.array(ys)

X, y = create_graph_sequences(volatility_scaled, SEQ_LENGTH)

# Μετατροπή shape από [Batch, Time, Nodes] -> [Batch, Time, Nodes, 1 Feature]
X = np.expand_dims(X, axis=-1)

# Split 80/20
split_idx = int(len(X) * 0.8)
X_train, X_test = torch.FloatTensor(X[:split_idx]), torch.FloatTensor(X[split_idx:])
y_train, y_test = torch.FloatTensor(y[:split_idx]), torch.FloatTensor(y[split_idx:])

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=False)

print(f"X_train (GNN shape): {X_train.shape} -> [Batch, Time, Nodes, Features]")
print(f"y_train (GNN target): {y_train.shape} -> [Batch, Nodes]")

X_train (GNN shape): torch.Size([2377, 14, 10, 1]) -> [Batch, Time, Nodes, Features]
y_train (GNN target): torch.Size([2377, 10]) -> [Batch, Nodes]


In [5]:
# ==============================================================================
# ΑΡΧΙΤΕΚΤΟΝΙΚΕΣ GNN: GCN+LSTM και GAT+GRU
# ==============================================================================
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, GATConv

# --- 1. GCN + LSTM (Το κλασικό T-GCN) ---
class TGCN_LSTM(torch.nn.Module):
    def __init__(self, num_nodes, in_channels=1, hidden_dim=32, out_channels=1):
        super(TGCN_LSTM, self).__init__()
        self.num_nodes = num_nodes
        self.hidden_dim = hidden_dim
        
        self.gcn = GCNConv(in_channels, hidden_dim)
        self.lstm = torch.nn.LSTM(hidden_dim, hidden_dim, batch_first=True)
        self.fc = torch.nn.Linear(hidden_dim, out_channels)

    def forward(self, x, edge_index, edge_weight=None):
        batch_size, seq_len, nodes, _ = x.shape
        
        spatial_seq = []
        for t in range(seq_len):
            x_t = x[:, t, :, :].reshape(-1, 1) 
            h_t = F.relu(self.gcn(x_t, edge_index, edge_weight))
            h_t = h_t.reshape(batch_size, nodes, self.hidden_dim)
            spatial_seq.append(h_t)
            
        spatial_seq = torch.stack(spatial_seq, dim=1) # [Batch, Time, Nodes, Hidden]
        lstm_input = spatial_seq.permute(0, 2, 1, 3).reshape(batch_size * nodes, seq_len, self.hidden_dim)
        
        lstm_out, _ = self.lstm(lstm_input)
        out = self.fc(lstm_out[:, -1, :]) # Τελευταίο Time-Step
        out = out.reshape(batch_size, nodes)
        
        return F.softplus(out) 

# --- 2. GAT + GRU ---
class TGAT_GRU(torch.nn.Module):
    def __init__(self, num_nodes, in_channels=1, hidden_dim=32, out_channels=1):
        super(TGAT_GRU, self).__init__()
        self.num_nodes = num_nodes
        self.hidden_dim = hidden_dim
        
        # Το GAT βρίσκει δυναμικά βάρη ανάμεσα στους γείτονες
        self.gat = GATConv(in_channels, hidden_dim, heads=1) 
        self.gru = torch.nn.GRU(hidden_dim, hidden_dim, batch_first=True)
        self.fc = torch.nn.Linear(hidden_dim, out_channels)

    def forward(self, x, edge_index, edge_weight=None):
        batch_size, seq_len, nodes, _ = x.shape
        
        spatial_seq = []
        for t in range(seq_len):
            x_t = x[:, t, :, :].reshape(-1, 1)
            # Το GAT δεν χρησιμοποιεί edge_weight, μαθαίνει τα δικά του βάρη
            h_t = F.relu(self.gat(x_t, edge_index))
            h_t = h_t.reshape(batch_size, nodes, self.hidden_dim)
            spatial_seq.append(h_t)
            
        spatial_seq = torch.stack(spatial_seq, dim=1)
        gru_input = spatial_seq.permute(0, 2, 1, 3).reshape(batch_size * nodes, seq_len, self.hidden_dim)
        
        gru_out, _ = self.gru(gru_input)
        out = self.fc(gru_out[:, -1, :])
        out = out.reshape(batch_size, nodes)
        
        return F.softplus(out)

print("GCN+LSTM και GAT+GRU δημιουργήθηκαν επιτυχώς!")

GCN+LSTM και GAT+GRU δημιουργήθηκαν επιτυχώς!


In [6]:
# ==============================================================================
# ΕΚΠΑΙΔΕΥΣΗ GNN MODELS ΚΑΙ ΕΞΑΓΩΓΗ METRICS
# ==============================================================================
from sklearn.metrics import mean_squared_error, mean_absolute_error

def train_gnn(model, name, edge_idx, edge_wt, epochs=50):
    print(f"\n--- Εκπαίδευση {name} ---")
    optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
    criterion = torch.nn.MSELoss()
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for b_x, b_y in train_loader:
            optimizer.zero_grad()
            out = model(b_x, edge_idx, edge_wt)
            loss = criterion(out, b_y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            
        if (epoch+1) % 10 == 0:
            print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f}")
    return model

# 1. Εκπαίδευση
model_gcn_lstm = TGCN_LSTM(num_nodes)
model_gat_gru = TGAT_GRU(num_nodes)

train_gnn(model_gcn_lstm, "GCN+LSTM (Spillover Graph)", edge_index_spill, edge_weight_spill, epochs=40)
train_gnn(model_gat_gru, "GAT+GRU (Spillover Graph)", edge_index_spill, None, epochs=40)

# 2. Αξιολόγηση
def evaluate_gnn(model, name, edge_idx, edge_wt):
    model.eval()
    with torch.no_grad():
        y_pred = model(X_test, edge_idx, edge_wt).numpy()
    
    # Παίρνουμε μόνο το Node 0 (BTC) και κάνουμε Inverse Transform
    y_pred_btc = scaler.inverse_transform(y_pred)[:, 0] * 100
    y_true_btc = scaler.inverse_transform(y_test.numpy())[:, 0] * 100
    y_pred_btc = np.clip(y_pred_btc, 0.1, None)
    
    rmse = np.sqrt(mean_squared_error(y_true_btc, y_pred_btc))
    mae = mean_absolute_error(y_true_btc, y_pred_btc)
    qlike = np.mean(np.log(y_pred_btc**2) + (y_true_btc**2 / y_pred_btc**2))
    
    return {'Model': name, 'RMSE': rmse, 'MAE': mae, 'QLIKE': qlike}

res_1 = evaluate_gnn(model_gcn_lstm, "GCN+LSTM", edge_index_spill, edge_weight_spill)
res_2 = evaluate_gnn(model_gat_gru, "GAT+GRU", edge_index_spill, None)

df_gnn = pd.DataFrame([res_1, res_2]).set_index('Model')
print("\n=== ST-GNN BENCHMARK (TEST SET - BTC) ===")
print(df_gnn.round(4))


--- Εκπαίδευση GCN+LSTM (Spillover Graph) ---
Epoch 10/40 | Loss: 0.0048
Epoch 20/40 | Loss: 0.0044
Epoch 30/40 | Loss: 0.0041
Epoch 40/40 | Loss: 0.0041

--- Εκπαίδευση GAT+GRU (Spillover Graph) ---
Epoch 10/40 | Loss: 0.0049
Epoch 20/40 | Loss: 0.0044
Epoch 30/40 | Loss: 0.0041
Epoch 40/40 | Loss: 0.0040

=== ST-GNN BENCHMARK (TEST SET - BTC) ===
            RMSE     MAE   QLIKE
Model                           
GCN+LSTM  1.7579  1.2695  3.0408
GAT+GRU   1.7528  1.2629  3.0269


In [7]:
# ==============================================================================
# ΜΗΝΑΣ 4 (ROBUSTNESS CHECK): GNN EVALUATION ΣΕ BINANCE ΚΑΙ COINBASE
# ==============================================================================

print("\n=== ROBUSTNESS CHECK ST-GNN (BINANCE & COINBASE) ===")

def prepare_gnn_external_data(filepath):
    df = pd.read_csv(filepath, index_col='Date', parse_dates=True)
    # Βάζουμε 0 όπου λείπει νόμισμα (NaN)
    # Έτσι, οι κόμβοι (π.χ. BNB στο Coinbase) απλώς δίνουν σήμα "μηδενικής δραστηριότητας"
    # αντί να "σπάσουν" τον Γράφο.
    df = df.fillna(0)
    
    vol_data = np.abs(df.values)
    vol_scaled = scaler.transform(vol_data) # Χρήση του Yahoo Scaler
    
    X_ext, y_ext = create_graph_sequences(vol_scaled, SEQ_LENGTH)
    X_ext = np.expand_dims(X_ext, axis=-1)
    
    return torch.FloatTensor(X_ext), torch.FloatTensor(y_ext)

# Φόρτωση
X_bin, y_bin = prepare_gnn_external_data('crypto_log_returns_binance.csv')
X_coin, y_coin = prepare_gnn_external_data('crypto_log_returns_coinbase.csv')

def eval_frozen_gnn(X_tens, y_tens, source_name):
    print(f"\n--- ST-GNNs σε {source_name} ---")
    model_gcn_lstm.eval()
    with torch.no_grad():
        y_pred = model_gcn_lstm(X_tens, edge_index_spill, edge_weight_spill).numpy()
        
    y_pred_btc = scaler.inverse_transform(y_pred)[:, 0] * 100
    y_true_btc = scaler.inverse_transform(y_tens.numpy())[:, 0] * 100
    y_pred_btc = np.clip(y_pred_btc, 0.1, None)
    
    rmse = np.sqrt(mean_squared_error(y_true_btc, y_pred_btc))
    qlike = np.mean(np.log(y_pred_btc**2) + (y_true_btc**2 / y_pred_btc**2))
    
    print(f"GCN+LSTM | RMSE: {rmse:.4f} | QLIKE: {qlike:.4f}")

eval_frozen_gnn(X_bin, y_bin, "BINANCE")
eval_frozen_gnn(X_coin, y_coin, "COINBASE")


=== ROBUSTNESS CHECK ST-GNN (BINANCE & COINBASE) ===

--- ST-GNNs σε BINANCE ---
GCN+LSTM | RMSE: 1.7484 | QLIKE: 3.0910

--- ST-GNNs σε COINBASE ---
GCN+LSTM | RMSE: 1.6532 | QLIKE: 2.8470
